In [2]:
!pip install transformers torch

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [4]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

In [6]:
# Initialize chat history
chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.\n")

while True:
    # Take user input
    user_input = input("You: ")

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Have a good day! 👋")
        break

    # Expand short forms (improves accuracy)
    if user_input.lower() == "ml":
        user_input = "What is Machine Learning?"
    elif user_input.lower() == "ai":
        user_input = "What is Artificial Intelligence?"

    # Better prompt formatting
    formatted_input = "User: " + user_input + "\nBot:"

    # Tokenize input
    new_input = tokenizer(formatted_input + tokenizer.eos_token, return_tensors='pt')
    new_input_ids = new_input['input_ids']
    attention_mask = new_input['attention_mask']

    # Append chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)

        # Create attention mask for full conversation
        attention_mask = torch.cat(
            [torch.ones(chat_history_ids.shape, dtype=torch.long), attention_mask],
            dim=-1
        )
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=300,
        do_sample=True,
        top_k=40,
        top_p=0.85,
        temperature=0.7,        # reduces randomness
        repetition_penalty=1.2, # avoids repeated nonsense
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode response
    bot_response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    # 🔹 Fallback for bad responses
    if len(bot_response.strip()) < 2:
        bot_response = "I'm not sure I understood that. Could you rephrase?"

    print("Chatbot:", bot_response, "\n")

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.



You:  Hello


Chatbot: Hi! I am a receiving bot. Please message me if you have any questions or concerns. Thanks! 



You:  What is ML?


Chatbot: Machine learning, it's what we're all talking about here. It's just that some people don't like to use it because they think it sounds like a lot of work. 



You:  What is AI?


Chatbot: It's an artificial intelligence that comes from the future and does things. 



You:  EXIT


Chatbot: Have a good day! 👋
